In [2]:
!pip install osmnx

In [ ]:
import os
import re
import time
import math
import json
import requests
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx

from shapely.geometry import LineString, Point, MultiLineString
from shapely.ops import linemerge

warnings.filterwarnings("ignore", category=UserWarning)

Matplotlib is building the font cache; this may take a moment.


## __Paths__

In [ ]:
OUT_PATH = r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/Pruebas OSM-OSMNX/osmnx-AMG/Networks 01'

## __Helpers__

In [ ]:
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def normalize_text(s):
    if pd.isna(s) or s is None:
        return None
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def save_gdf(gdf, path):
    if gdf is None or gdf.empty:
        print(f"[WARN] Empty GDF not saved: {path}")
        return
    ensure_dir(os.path.dirname(path))
    gdf.to_file(path)
    print(f"[OK] Saved: {path}")

def save_csv(df, path):
    ensure_dir(os.path.dirname(path))
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[OK] Saved: {path}")

def overpass_query(query, pause=1.5):
    r = requests.get(OVERPASS_URL, params={"data": query}, timeout=240)
    r.raise_for_status()
    time.sleep(pause)
    return r.json()

def polygon_to_overpass_poly_string(geom):
    if geom.geom_type == "MultiPolygon":
        geom = geom.convex_hull
    elif geom.geom_type != "Polygon":
        geom = geom.convex_hull
    coords = list(geom.exterior.coords)
    # Overpass poly: "lat lon lat lon ..."
    return " ".join([f"{y} {x}" for x, y in coords])

def graph_to_nodes_edges(G):
    nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True)
    if nodes.crs is None:
        nodes = nodes.set_crs(WGS84)
    if edges.crs is None:
        edges = edges.set_crs(WGS84)
    return nodes, edges

def parse_overpass_elements(data):
    nodes = {}
    ways = {}
    relations = {}
    for el in data.get("elements", []):
        typ = el.get("type")
        if typ == "node":
            nodes[el["id"]] = el
        elif typ == "way":
            ways[el["id"]] = el
        elif typ == "relation":
            relations[el["id"]] = el
    return nodes, ways, relations

def get_way_nodes(way):
    return way.get("nodes", []) if way else []

def way_to_linestring(way, node_lookup):
    if "geometry" in way and way["geometry"]:
        coords = [(p["lon"], p["lat"]) for p in way["geometry"]]
        if len(coords) >= 2:
            return LineString(coords)

    coords = []
    for nid in way.get("nodes", []):
        n = node_lookup.get(nid)
        if n and "lon" in n and "lat" in n:
            coords.append((n["lon"], n["lat"]))
    if len(coords) >= 2:
        return LineString(coords)
    return None

def length_m(geom):
    if geom is None or geom.is_empty:
        return np.nan
    s = gpd.GeoSeries([geom], crs=WGS84).to_crs(METRIC_CRS)
    return float(s.length.iloc[0])

def flatten_osmid(val):
    if isinstance(val, list):
        return val
    return [val]

def classify_allowed_modes(row):
    """
    Etiqueta simple para que luego la traduzcas a TSysSet / permitidos en Visum.
    """
    if row.get("net_type") == "rail":
        return f"{TSYS_RAIL};{TSYS_WALK}"
    hw = str(row.get("highway", "")).lower()

    # simple heuristic
    if row.get("net_type") == "drive":
        return f"{TSYS_CAR};{TSYS_BUS}"
    elif row.get("net_type") == "all":
        return f"{TSYS_WALK};{TSYS_BUS};{TSYS_CAR}"
    return f"{TSYS_WALK}"

def one_way_flag(row):
    # OSMnx drive graph is directed; use simple 1/0 export
    ow = row.get("oneway", None)
    if pd.isna(ow) or ow is None:
        return 0
    if isinstance(ow, bool):
        return int(ow)
    if str(ow).lower() in ["yes", "true", "1"]:
        return 1
    return 0


## __Read AMG Polygon__

In [4]:
AMG_Polygon = gpd.read_file(r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/Importacion TransCAD/Limites Municipales/AMG/AMG_SHAPE.shp')
print("original AMG_Polygon CRS:", AMG_Polygon.crs)
AMG_Polygon = AMG_Polygon.to_crs(epsg=4326)
print("AMG_Polygon CRS after transformation:", AMG_Polygon.crs)

original AMG_Polygon CRS: EPSG:4019
AMG_Polygon CRS after transformation: EPSG:4326


## __Download networks__

In [5]:
# 2.1 Drive simplified (solo para apoyo / visual)
G_drive = ox.graph_from_polygon(
    AMG_Polygon.geometry.iloc[0],
    network_type="drive",
    simplify=True,
    retain_all=False,
    truncate_by_edge=False
)


# 2.2 All simplified (apoyo)
G_all = ox.graph_from_polygon(
    AMG_Polygon.geometry.iloc[0],
    network_type="all",
    simplify=True,
    retain_all=False,
    truncate_by_edge=False
)

# 2.4 Railway unsimplified para Guadalajara
rail_filter = (
    '["railway"~"light_rail|subway|rail"]'
    '["name"~"Línea 1 del Tren Eléctrico Urbano|'
    'Linea 1 del Tren Eléctrico Urbano|'
    'Línea 2 del Tren Eléctrico Urbano|'
    'Linea 2 del Tren Eléctrico Urbano|'
    'Línea 3 del Tren Eléctrico Urbano|'
    'Linea 3 del Tren Eléctrico Urbano|'
    'Línea 4 del Tren Eléctrico Urbano de Guadalajara|'
    'Linea 4 del Tren Eléctrico Urbano de Guadalajara"]'
)

G_rail_uns = ox.graph_from_polygon(
    AMG_Polygon.geometry.iloc[0],
    custom_filter=rail_filter,
    simplify=False, #AQUI si false para mantener todas las estaciones
    retain_all=True, #TRUE porque las 4 lineas no estan conectadas entre si
    truncate_by_edge=False
)

## __Compose Graphs__

In [7]:
G_base = nx.compose(G_all, G_rail_uns)

## __Save SHPs__

In [ ]:

#Get nodes and edges for each graph
drive_nodes, drive_edges = graph_to_nodes_edges(G_drive)
all_nodes, all_edges = graph_to_nodes_edges(G_all)
rail_nodes, rail_edges = graph_to_nodes_edges(G_rail_uns)
base_nodes, base_edges = graph_to_nodes_edges(G_base)


# tag source net
drive_edges["net_type"] = "drive"
all_edges["net_type"] = "all"
rail_edges["net_type"] = "rail"
base_edges["net_type"] = np.where(base_edges.get("railway").notna(), "rail", "all")

#Save outputs
save_gdf(drive_nodes, os.path.join(OUT_PATH, "drive_nodes.shp"))
save_gdf(drive_edges, os.path.join(OUT_PATH, "drive_edges.shp"))
save_gdf(all_nodes, os.path.join(OUT_PATH, "all_nodes.shp"))
save_gdf(all_edges, os.path.join(OUT_PATH, "all_edges.shp"))
save_gdf(rail_nodes, os.path.join(OUT_PATH, "rail_nodes.shp"))
save_gdf(rail_edges, os.path.join(OUT_PATH, "rail_edges.shp"))
save_gdf(base_nodes, os.path.join(OUT_PATH, "base_nodes.shp"))
save_gdf(base_edges, os.path.join(OUT_PATH, "base_edges.shp"))

## __Create Visum Node & Link Tables__

In [ ]:
# NODES
visum_nodes = base_nodes.reset_index().copy()

if "osmid" not in visum_nodes.columns:
    visum_nodes = visum_nodes.rename(columns={visum_nodes.columns[0]: "osmid"})

visum_nodes["NodeNo"] = range(1, len(visum_nodes) + 1)
visum_nodes["XCoord"] = visum_nodes.geometry.x
visum_nodes["YCoord"] = visum_nodes.geometry.y

node_no_map = dict(zip(visum_nodes["osmid"], visum_nodes["NodeNo"]))

visum_nodes_out = visum_nodes[["NodeNo", "osmid", "XCoord", "YCoord", "geometry"]].copy()


# LINKS
visum_links = base_edges.reset_index().copy()

# assign node numbers
visum_links["FromNodeNo"] = visum_links["u"].map(node_no_map)
visum_links["ToNodeNo"] = visum_links["v"].map(node_no_map)

# robust osmid handling
visum_links["osmid_list"] = visum_links["osmid"].apply(flatten_osmid)
visum_links["OSMWayID_primary"] = visum_links["osmid_list"].apply(lambda x: x[0] if len(x) > 0 else None)

visum_links["LinkNo"] = range(1, len(visum_links) + 1)
visum_links["Length_m"] = visum_links.to_crs(METRIC_CRS).length
visum_links["AllowedModes"] = visum_links.apply(classify_allowed_modes, axis=1)
visum_links["OneWay"] = visum_links.apply(one_way_flag, axis=1)
visum_links["Name"] = visum_links.get("name", None).apply(
    lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x
)


# table for lookup by edge triple
edgekey_to_linkno = {
    (row.u, row.v, row.key): row.LinkNo
    for row in visum_links.itertuples()
}

# helper index by osmid
osmid_to_edge_records = defaultdict(list)
for row in visum_links.itertuples():
    for oid in row.osmid_list:
        osmid_to_edge_records[oid].append({
            "u": row.u,
            "v": row.v,
            "key": row.key,
            "LinkNo": row.LinkNo,
            "FromNodeNo": row.FromNodeNo,
            "ToNodeNo": row.ToNodeNo,
            "geometry": row.geometry
        })

visum_links_out = visum_links[
    [
        "LinkNo", "u", "v", "key",
        "FromNodeNo", "ToNodeNo",
        "OSMWayID_primary", "Name",
        "highway", "railway", "net_type",
        "OneWay", "AllowedModes", "Length_m", "geometry"
    ]
].copy()

save_gdf(visum_nodes_out, os.path.join(OUT_PATH, "02_visum_ready", "visum_nodes.shp"))
save_gdf(visum_links_out, os.path.join(OUT_PATH, "02_visum_ready", "visum_links.shp"))
save_csv(
    visum_nodes_out.drop(columns="geometry"),
    os.path.join(OUT_PATH, "02_visum_ready", "visum_nodes.csv")
)
save_csv(
    visum_links_out.drop(columns="geometry"),
    os.path.join(OUT_PATH, "02_visum_ready", "visum_links.csv")
)

## __Get Bus Stops__

In [8]:
tags_stops = {
    "highway": "bus_stop",
    "public_transport": ["platform", "stop_position"],
}

bus_stops = ox.features_from_polygon(AMG_Polygon.geometry.iloc[0], tags_stops)


In [19]:
# keep point features only for clean Stop/StopArea/StopPoint creation
bus_stops = bus_stops[bus_stops.geometry.geom_type == "Point"].copy()
bus_stops

geometry                name  \
element id                                                            
node    8743369015  POINT (-103.29622 20.63216)     Lázaro Cárdenas   
        8743369016  POINT (-103.29617 20.63219)     Lázaro Cárdenas   
        9485209456  POINT (-103.30002 20.63773)  Tlaquepaque Centro   
        9485209457  POINT (-103.30005 20.63771)  Tlaquepaque Centro   
        9485209470   POINT (-103.3041 20.64482)            Río Nilo   
...                                         ...                 ...   
        9859495306  POINT (-103.31983 20.62913)     Calzada Córdova   
        9859495307  POINT (-103.31902 20.63102)          Xochimilco   
        9859495308  POINT (-103.31347 20.64472)              Hornos   
        9859495309   POINT (-103.31125 20.6463)        Av. Río Nilo   
        9859495310    POINT (-103.3106 20.6499)      Querido Moheno   

                          network operator public_transport railway subway  \
element id                                                                   
node    8743369015        Mi Tren   SITEUR    stop_position    stop    yes   
        8743369016        Mi Tren   SITEUR    stop_position    stop    yes   
        9485209456        Mi Tren   SITEUR    stop_position    stop    yes   
        9485209457        Mi Tren   SITEUR    stop_position    stop    yes   
        9485209470        Mi Tren   SITEUR    stop_position    stop    yes   
...                           ...      ...              ...     ...    ...   
        9859495306  Mi Transporte   Setran         platform     NaN    NaN   
        9859495307  Mi Transporte   Setran         platform     NaN    NaN   
        9859495308  Mi Transporte   Setran         platform     NaN    NaN   
        9859495309  Mi Transporte   Setran         platform     NaN    NaN   
        9859495310  Mi Transporte   Setran         platform     NaN    NaN   

                    bus                 gtfs_id   highway  \
element id                                                  
node    8743369015  NaN                     NaN       NaN   
        8743369016  NaN                     NaN       NaN   
        9485209456  NaN                     NaN       NaN   
        9485209457  NaN                     NaN       NaN   
        9485209470  NaN                     NaN       NaN   
...                 ...                     ...       ...   
        9859495306  yes       mxc_C108V2_STP_33  bus_stop   
        9859495307  yes       mxc_C108V2_STP_32  bus_stop   
        9859495308  yes          mxc_C05_STP_58  bus_stop   
        9859495309  yes  mxd_mxc_AMG_T02_STP_79  bus_stop   
        9859495310  yes  mxd_mxc_AMG_T02_STP_78  bus_stop   

                                       ref route_ref bench  bin shelter  lit  \
element id                                                                     
node    8743369015                     NaN       NaN   NaN  NaN     NaN  NaN   
        8743369016                     NaN       NaN   NaN  NaN     NaN  NaN   
        9485209456                     NaN       NaN   NaN  NaN     NaN  NaN   
        9485209457                     NaN       NaN   NaN  NaN     NaN  NaN   
        9485209470                     NaN       NaN   NaN  NaN     NaN  NaN   
...                                    ...       ...   ...  ...     ...  ...   
        9859495306       mxc_C108V2_STP_33       T02   NaN  NaN     NaN  NaN   
        9859495307       mxc_C108V2_STP_32       T02   NaN  NaN     NaN  NaN   
        9859495308          mxc_C05_STP_58       T02   NaN  NaN     NaN  NaN   
        9859495309  mxd_mxc_AMG_T02_STP_79       T02   NaN  NaN     NaN  NaN   
        9859495310  mxd_mxc_AMG_T02_STP_78       T02   NaN  NaN     NaN  NaN   

                   check_date:bin network:wikidata check_date:bench  \
element id                                                            
node    8743369015            NaN              NaN              NaN   
        8743369016            NaN              NaN  

## __Get Bus Stops with OSMNX bbox__

In [12]:
# =========================================================
# 1) LOAD POLYGON
# =========================================================

target_crs = "EPSG:4326"
metric_crs = "EPSG:32613"  # Guadalajara aprox UTM 13N
polygon_path = r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/Importacion TransCAD/Limites Municipales/AMG/AMG_SHAPE.shp'


AMG_Polygon = gpd.read_file(polygon_path).to_crs(target_crs)

# convertir a CRS métrico
AMG_metric = AMG_Polygon.to_crs(metric_crs)

# centroide y buffer de 2 km
centroid = AMG_metric.geometry.iloc[0].centroid
study_polygon_small = gpd.GeoSeries([centroid.buffer(2000)], crs=metric_crs).to_crs(target_crs).iloc[0]

dummy_area = gpd.GeoDataFrame(geometry=[study_polygon_small], crs=target_crs)

# bbox para Overpass
minx, miny, maxx, maxy = study_polygon_small.bounds
bbox = f"{miny},{minx},{maxy},{maxx}"

print("Using dummy bbox:", bbox)


Using dummy bbox: 20.614925361691775,-103.3294974078383,20.651053704696455,-103.29111937703983


In [14]:
tags_stops = {
    "highway": "bus_stop",
    "public_transport": ["platform", "stop_position"],
}
bbox = (minx, miny, maxx, maxy)
bus_stops = ox.features_from_bbox(bbox, tags_stops)

In [16]:
bus_stops = bus_stops[bus_stops.geometry.geom_type == "Point"].copy()
bus_stops

geometry                name  \
element id                                                            
node    8743369015  POINT (-103.29622 20.63216)     Lázaro Cárdenas   
        8743369016  POINT (-103.29617 20.63219)     Lázaro Cárdenas   
        9485209456  POINT (-103.30002 20.63773)  Tlaquepaque Centro   
        9485209457  POINT (-103.30005 20.63771)  Tlaquepaque Centro   
        9485209470   POINT (-103.3041 20.64482)            Río Nilo   
...                                         ...                 ...   
        9859495306  POINT (-103.31983 20.62913)     Calzada Córdova   
        9859495307  POINT (-103.31902 20.63102)          Xochimilco   
        9859495308  POINT (-103.31347 20.64472)              Hornos   
        9859495309   POINT (-103.31125 20.6463)        Av. Río Nilo   
        9859495310    POINT (-103.3106 20.6499)      Querido Moheno   

                          network operator public_transport railway subway  \
element id                                                                   
node    8743369015        Mi Tren   SITEUR    stop_position    stop    yes   
        8743369016        Mi Tren   SITEUR    stop_position    stop    yes   
        9485209456        Mi Tren   SITEUR    stop_position    stop    yes   
        9485209457        Mi Tren   SITEUR    stop_position    stop    yes   
        9485209470        Mi Tren   SITEUR    stop_position    stop    yes   
...                           ...      ...              ...     ...    ...   
        9859495306  Mi Transporte   Setran         platform     NaN    NaN   
        9859495307  Mi Transporte   Setran         platform     NaN    NaN   
        9859495308  Mi Transporte   Setran         platform     NaN    NaN   
        9859495309  Mi Transporte   Setran         platform     NaN    NaN   
        9859495310  Mi Transporte   Setran         platform     NaN    NaN   

                    bus                 gtfs_id   highway  \
element id                                                  
node    8743369015  NaN                     NaN       NaN   
        8743369016  NaN                     NaN       NaN   
        9485209456  NaN                     NaN       NaN   
        9485209457  NaN                     NaN       NaN   
        9485209470  NaN                     NaN       NaN   
...                 ...                     ...       ...   
        9859495306  yes       mxc_C108V2_STP_33  bus_stop   
        9859495307  yes       mxc_C108V2_STP_32  bus_stop   
        9859495308  yes          mxc_C05_STP_58  bus_stop   
        9859495309  yes  mxd_mxc_AMG_T02_STP_79  bus_stop   
        9859495310  yes  mxd_mxc_AMG_T02_STP_78  bus_stop   

                                       ref route_ref bench  bin shelter  lit  \
element id                                                                     
node    8743369015                     NaN       NaN   NaN  NaN     NaN  NaN   
        8743369016                     NaN       NaN   NaN  NaN     NaN  NaN   
        9485209456                     NaN       NaN   NaN  NaN     NaN  NaN   
        9485209457                     NaN       NaN   NaN  NaN     NaN  NaN   
        9485209470                     NaN       NaN   NaN  NaN     NaN  NaN   
...                                    ...       ...   ...  ...     ...  ...   
        9859495306       mxc_C108V2_STP_33       T02   NaN  NaN     NaN  NaN   
        9859495307       mxc_C108V2_STP_32       T02   NaN  NaN     NaN  NaN   
        9859495308          mxc_C05_STP_58       T02   NaN  NaN     NaN  NaN   
        9859495309  mxd_mxc_AMG_T02_STP_79       T02   NaN  NaN     NaN  NaN   
        9859495310  mxd_mxc_AMG_T02_STP_78       T02   NaN  NaN     NaN  NaN   

                   check_date:bin network:wikidata check_date:bench  \
element id                                                            
node    8743369015            NaN              NaN              NaN   
        8743369016            NaN              NaN  